# Working within a factor of the best algorithm. 

For this part, I will use the data that I have already collected regarding the algo that has the best ERT. 

This startegy is a smart approach to keeping in mind algorithms that behave similarly. This is method is beneficial for algorithms that have are slightly less performing than the best algorithm, but still remain interesting to investigate their performance. This contrasts the other method where only the best algo was kept and all other algorithms were discarded. 
## 1) Comparing the best ERT of each dimension, function, target with the algos from all the different years. 

In this part, I will read the bestERT from the file global_best_algos.csv. This bestERT is determinant in the definition of the best algorithms and will determine which algos I can keep. 

We will define a factor (ex: 2), and we will keep all the algorithms that have an ERT that is within 2 times the ERT of the overall bestERT. This is done over every dimension, function and target.  

The output result of this will give us a CSV file tracking which are these good algorithms for each dimension, function and target precision. 

These are the following steps I did to implement the code:

1) Read the csv file where the best algos appear, and all the information regarding dimension, function and target 
2) Loop over every year. 
3) Create an interactive interface where we ask the user what factor they want. 
4) Inside every year, loop over dimension function and target and check if the ERT is within the factor bounds. 
5) If it is: write the name of the algo, the year, the dimension, the function and the target to a CSV file. 
6) If it isn't then pass and keep going to the next algo.

In [2]:
import pandas as pd
import numpy as np
import cocopp


# ============================================================
# 1. LOAD YOUR CSV WITH BEST ERTs  
# ============================================================

# make sure to change the file path accordding to what test suite you are working with 

df_best = pd.read_csv("results/global_best_algos_bbob-noisy.csv")

# clean
df_best = df_best[df_best.best_algorithm.notna()]
df_best = df_best[df_best.best_ERT.notna()]

# all known targets (sorted, unique)
all_targets = sorted(df_best["target"].unique())

# dictionary: (dim, func, target) → best_ERT
best_dict = {}
for _, row in df_best.iterrows():
    key = (int(row.dimension), int(row.function_id), float(row.target))
    best_dict[key] = float(row.best_ERT)


# ============================================================
# 2. ASK USER FOR FACTOR
# ============================================================

try:
    factor = float(input("Enter tolerance factor (default=2): ") or 2)
except:
    factor = 2.0

print(f"\nUsing factor = {factor}\n")


# ============================================================
# 3. LOAD COCO DATA BY YEAR
# ============================================================
# again make sure to select the right file path for the corresponding test suite 

years = df_best["year"].unique()     # automatically detect existing years
years = sorted(years)

dsl_by_year = {}

for y in years:
    print(f"Loading BBOB noisy data for year {y} ...")
    try:
        dsl_by_year[y] = cocopp.load(f"bbob-noisy/{y}/*")
    except:
        print(f" Warning: Could not load year {y}. Skipping.")
        continue


# ============================================================
# 4. LOOP USING YOUR EXACT CODE STRUCTURE
# ============================================================

within_rows = []

for year, dsl in dsl_by_year.items():
    
    print(f"\n=== Processing year {year} ===")
    
    dd = dsl.dictByDimFunc()     # same as in your working code

    for dim in sorted(dd.keys()):
        for func in sorted(dd[dim].keys()):
            
            # loop over algorithms
            for ds in dd[dim][func]:
                
                algo = ds.algId

                # loop over targets from your CSV
                for tgt in all_targets:

                    key = (dim, func, tgt)
                    if key not in best_dict:
                        continue     # best ERT not known for this triple

                    best_ert = best_dict[key]

                    # compute ERT for THIS algorithm at THIS target
                    try:
                        ert = float(ds.detERT([tgt])[0])
                    except Exception:
                        continue

                    if not np.isfinite(ert):
                        continue

                    # check whether ERT is within factor
                    if ert <= factor * best_ert:
                        within_rows.append([
                            year,
                            dim,
                            func,
                            tgt,
                            algo,
                            ert,
                            best_ert
                        ])


# ============================================================
# 5. SAVE OUTPUT CSV
# ============================================================

within_df = pd.DataFrame(
    within_rows,
    columns=[
        "year",
        "dimension",
        "function_id",
        "target",
        "algorithm",
        "ERT",
        "best_ERT"
    ]
)

output_path = "results/within_factor_algorithms_bbob-noisy.csv"
within_df.to_csv(output_path, index=False)

print(f"\n✓ Saved: {output_path}")
print(f"Total rows: {len(within_df)}")


Enter tolerance factor (default=2): 2

Using factor = 2.0

Loading BBOB noisy data for year 2009 ...
Loading BBOB noisy data for year 2010 ...
Loading BBOB noisy data for year 2012 ...
Loading BBOB noisy data for year 2016 ...

=== Processing year 2009 ===

=== Processing year 2010 ===

=== Processing year 2012 ===

=== Processing year 2016 ===

✓ Saved: results/within_factor_algorithms_bbob-noisy.csv
Total rows: 3744


## 2) Counting the number of times an algo appears for each dimension

In this part, we are going to be aggregating over all the different function and targets within a certain dimension. The goal is to have for each dimension the best performing algorithms with the factor determined above, and also display the count.

In [3]:
import pandas as pd

# ============================================================
# 1. READ THE CSV FROM PART 1
# ============================================================

df = pd.read_csv("results/within_factor_algorithms_bbob-noisy.csv")

# clean potential NaNs
df = df[df.algorithm.notna()]
df = df[df.ERT.notna()]


# ============================================================
# 2. INITIALIZE THE COUNTER
# ============================================================
# We use a dictionary: (dimension, algorithm) → count

counter = {}

for _, row in df.iterrows():
    dim  = int(row["dimension"])
    algo = row["algorithm"]

    key = (dim, algo)

    if key not in counter:
        counter[key] = 0
    counter[key] += 1


# ============================================================
# 3. CONVERT DICTIONARY TO DATAFRAME
# ============================================================

rows = []
for (dim, algo), count in counter.items():
    rows.append([dim, algo, count])

counts_df = pd.DataFrame(rows, columns=["dimension", "algorithm", "count"])


# Sort: first by dimension, then by descending count
counts_df = counts_df.sort_values(["dimension", "count"], ascending=[True, False])


# ============================================================
# 4. SAVE THE RESULTING CSV
# ============================================================

output_path = "results/counts_by_dimension_bbob-noisy.csv"
counts_df.to_csv(output_path, index=False)

print(f"\n✓ Saved: {output_path}")
print(f"Total rows: {len(counts_df)}")

# Optional: display a preview
print("\n=== Preview ===")
print(counts_df.head(20))



✓ Saved: results/counts_by_dimension_bbob-noisy.csv
Total rows: 183

=== Preview ===
     dimension                          algorithm  count
141          2         IPOPsaACM_loshchilov_noisy     78
76           2           IPOP-ACTCMA-ES_ros_noisy     68
83           2              IPOP-CMA-ES_ros_noisy     53
10           2          BIPOP-CMA-ES_hansen_noisy     45
142          2                  SNES_schaul_noisy     40
75           2      1komma4mirser_brockhoff_noisy     39
6            2                   GLOBAL_pal_noisy     37
7            2           MA-LS-CHAIN_molina_noisy     32
79           2         1komma4mir_brockhoff_noisy     31
0            2               FULLNEWUOA_ros_noisy     30
14           2                  ALPS_hornby_noisy     29
13           2          IPOP-SEP-CMA-ES_ros_noisy     28
164          2  PSAaLmC-CMA-ES_Nishida_bbob-noisy     27
1            2                    MCS_huyer_noisy     23
2            2                SNOBFIT_huyer_noisy     22
80

In [13]:
import pandas as pd

# ============================================================
# 1. LOAD THE COUNTS DATA
# ============================================================

df = pd.read_csv("results/counts_by_dimension_bbob-noisy.csv")

# Clean and sort
df = df.reset_index(drop=True)
df = df.sort_values(["dimension", "count"], ascending=[True, False])


# ============================================================
# 2. BUILD DICTIONARY: dim → list of (algo, count)
# ============================================================

ranking_dict = {}

for dim in sorted(df["dimension"].unique()):
    df_dim = df[df["dimension"] == dim]
    ranking_dict[dim] = list(zip(df_dim["algorithm"], df_dim["count"]))


# ============================================================
# 3. DETERMINE MAX RANK
# ============================================================

max_len = max(len(v) for v in ranking_dict.values())


# ============================================================
# 4. BUILD WIDE RANKING TABLE
# ============================================================

rows = []

for rank in range(max_len):
    row = {"rank": rank + 1}

    for dim in sorted(ranking_dict.keys()):
        if rank < len(ranking_dict[dim]):
            algo, count = ranking_dict[dim][rank]
            row[f"dim {dim}"] = f"{algo} ({count})"
        else:
            row[f"dim {dim}"] = ""
    
    rows.append(row)

ranking_table = pd.DataFrame(rows)


# ============================================================
# 5. DISPLAY NICELY USING PANDAS STYLING
# ============================================================

styled_table = (
    ranking_table.style
        .set_properties(**{
            "background-color": "#f7f7f7",
            "border": "1px solid #ccc",
            "padding": "6px",
            "font-size": "12px"
        })
        .set_table_styles([
            {"selector": "th", 
             "props": [("background-color", "#e6e6e6"),
                       ("font-weight", "bold"),
                       ("border", "1px solid #aaa"),
                       ("padding", "6px")]}
        ])
        .hide(axis="index")  # Hide the pandas index entirely
)

styled_table





rank,dim 2,dim 3,dim 5,dim 10,dim 20,dim 40
1,IPOPsaACM_loshchilov_noisy (78),IPOPsaACM_loshchilov_noisy (101),IPOPsaACM_loshchilov_noisy (97),IPOPsaACM_loshchilov_noisy (101),IPOPsaACM_loshchilov_noisy (115),IPOP-ACTCMA-ES_ros_noisy (102)
2,IPOP-ACTCMA-ES_ros_noisy (68),IPOP-ACTCMA-ES_ros_noisy (88),IPOP-ACTCMA-ES_ros_noisy (80),IPOP-ACTCMA-ES_ros_noisy (87),IPOP-ACTCMA-ES_ros_noisy (97),IPOP-CMA-ES_ros_noisy (69)
3,IPOP-CMA-ES_ros_noisy (53),IPOP-CMA-ES_ros_noisy (50),IPOP-CMA-ES_ros_noisy (53),IPOP-CMA-ES_ros_noisy (51),IPOP-CMA-ES_ros_noisy (51),BIPOP-CMA-ES_hansen_noisy (64)
4,BIPOP-CMA-ES_hansen_noisy (45),BIPOP-CMA-ES_hansen_noisy (47),BIPOP-CMA-ES_hansen_noisy (44),BIPOP-CMA-ES_hansen_noisy (44),BIPOP-CMA-ES_hansen_noisy (42),1komma4mirser_brockhoff_noisy (41)
5,SNES_schaul_noisy (40),PSAaLmC-CMA-ES_Nishida_bbob-noisy (41),PSAaLmC-CMA-ES_Nishida_bbob-noisy (37),PSAaSmD-CMA-ES_Nishida_bbob-noisy (39),1komma4mirser_brockhoff_noisy (42),IPOP-SEP-CMA-ES_ros_noisy (35)
6,1komma4mirser_brockhoff_noisy (39),1komma4mirser_brockhoff_noisy (38),PSAaLmD-CMA-ES_Nishida_bbob-noisy (31),PSAaSmC-CMA-ES_Nishida_bbob-noisy (37),PSAaSmC-CMA-ES_Nishida_bbob-noisy (32),SNES_schaul_noisy (10)
7,GLOBAL_pal_noisy (37),1komma4mir_brockhoff_noisy (31),1komma4mirser_brockhoff_noisy (25),1komma4mirser_brockhoff_noisy (35),1komma4ser_brockhoff_noisy (26),CMA-ESPLUSSEL_auger_noisy (7)
8,MA-LS-CHAIN_molina_noisy (32),SNES_schaul_noisy (27),PSAaSmC-CMA-ES_Nishida_bbob-noisy (23),PSAaLmD-CMA-ES_Nishida_bbob-noisy (23),1komma2mirser_brockhoff_noisy (23),MOS_torre_noisy (5)
9,1komma4mir_brockhoff_noisy (31),PSAaSmC-CMA-ES_Nishida_bbob-noisy (24),SNES_schaul_noisy (22),PSAaLmC-CMA-ES_Nishida_bbob-noisy (22),PSAaSmD-CMA-ES_Nishida_bbob-noisy (22),CMAEGS_finck_noisy (5)
10,FULLNEWUOA_ros_noisy (30),PSAaLmD-CMA-ES_Nishida_bbob-noisy (24),FULLNEWUOA_ros_noisy (18),1komma2mirser_brockhoff_noisy (21),1komma4mir_brockhoff_noisy (20),xNESas_schaul_noisy (5)


An interesting result is that the algo IPOPsaACM_loshchilov_noisy dominates in performance in almost all categories, except the last one (dim 40), where it is IPOP-ACTCMA-ES_ros_noisy

In [14]:
# Aggregate counts over all dimensions
overall_counts = (
    df.groupby("algorithm")["count"]
      .sum()
      .reset_index()
      .sort_values("count", ascending=False)
)

print("\nOverall frequency of being best (aggregated over ALL dimensions):\n")
print(overall_counts.head(10))



Overall frequency of being best (aggregated over ALL dimensions):

                            algorithm  count
20           IPOP-ACTCMA-ES_ros_noisy    522
23         IPOPsaACM_loshchilov_noisy    492
21              IPOP-CMA-ES_ros_noisy    327
13          BIPOP-CMA-ES_hansen_noisy    286
6       1komma4mirser_brockhoff_noisy    220
31  PSAaSmC-CMA-ES_Nishida_bbob-noisy    136
29  PSAaLmC-CMA-ES_Nishida_bbob-noisy    134
22          IPOP-SEP-CMA-ES_ros_noisy    126
37                  SNES_schaul_noisy    120
5          1komma4mir_brockhoff_noisy    117


In [15]:
overall_counts["percentage"] = (
    overall_counts["count"] / overall_counts["count"].sum() * 100
).round(2)

overall_counts.head(10)


,algorithm,count,percentage
20,IPOP-ACTCMA-ES_ros_noisy,522,13.94
23,IPOPsaACM_loshchilov_noisy,492,13.14
21,IPOP-CMA-ES_ros_noisy,327,8.73
13,BIPOP-CMA-ES_hansen_noisy,286,7.64
6,1komma4mirser_brockhoff_noisy,220,5.88
31,PSAaSmC-CMA-ES_Nishida_bbob-noisy,136,3.63
29,PSAaLmC-CMA-ES_Nishida_bbob-noisy,134,3.58
22,IPOP-SEP-CMA-ES_ros_noisy,126,3.37
37,SNES_schaul_noisy,120,3.21
5,1komma4mir_brockhoff_noisy,117,3.12


# 3) Plotting the results using the COCO platform tool

This last step is essential for checking if the result sin our table are indeed coherent. 
Using the COCO tool, we can access make a full analysis of the best performing algorithms. 

## 3.1) Performance of the best algorithms for each dimension

The first comparison between algorithms I want to do, is see the performance of the best algorithms for the different dimensionsion. There is a clear dominance of one algorihm as it appears as best in 5/6 dimensions.

On a side note, these 2 algos are also the best 2 for the dimension 2.

In [7]:
cocopp.main(['IPOPsaACM_loshchilov_noisy','IPOP-ACTCMA-ES_ros_noisy'])

Post-processing (2+)
  downloading https://numbbo.github.io/data-archive/data-archive/bbob-noisy/2012/IPOPsaACM_loshchilov_noisy.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz
  Using 2 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\IPOP-ACTCMA-ES_ros_noisy.tar.gz

Post-processing (2+)
  loading data...
    archive extracted to folder C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\.extracted_IPOPsaACM_loshchilov_noisy ...
  Data consistent according to consistency_check() in pproc.DataSet
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\IPOP-ACTCMA-ES_ros_noisy.tar.gz
  Will generate output data in folder ppdata\I

DictAlg([(('IPOPsaACM_loshchilov_noisy', ''),
          [DataSet(IPOPsaACM_loshchilov_noisy on f101 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f102 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f103 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f104 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f105 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f106 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f107 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f108 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f109 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f110 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f111 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f112 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f113 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f114 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f115 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f116 

In [ ]:
cocopp.main(['IPOPsaACM_loshchilov_noisy','IPOP-ACTCMA-ES_ros_noisy','IPOP-CMA-ES_ros_noisy','BIPOP-CMA-ES_hansen_noisy','1komma4mirser_brockhoff_noisy'])

Post-processing (2+)
  Using 5 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\IPOP-ACTCMA-ES_ros_noisy.tar.gz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\IPOP-CMA-ES_ros_noisy.tar.gz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2009\BIPOP-CMA-ES_hansen_noisy.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\1komma4mirser_brockhoff_noisy.tar.gz

Post-processing (2+)
  loading data...
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\IPOP-ACTCMA-ES_ros_noisy.tar.gz
  Data consistent according to consistency_check() in pproc.DataSet
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\IPOP-

If this type of error occurs:

Exception: There is more than a single entry associated with folder C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz on 2-D f101.


--> Go to the folder and delete it. 
--> When run the cocopp platform it will reinstall it.

We notice from the plots that these algorithms have a very similar behavior and almost overlap at every instance. 


## 3.2) Comparison of the 3 best algorithms for all dimensions

Here We observe that in almost all dimensions, the leaderboard for the top 3 algorithms is the same accross all dimensions except the last dimension 40. 


In [ ]:
cocopp.main(['IPOPsaACM_loshchilov_noisy','IPOP-ACTCMA-ES_ros_noisy','IPOP-CMA-ES_ros_noisy', 'BIPOP-CMA-ES_hansen_noisy',])